## Hospital Readmissions - Preprocess and EDA

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn import ensemble

In [3]:
# Load datasets

df_diabetic_data = pd.read_csv('diabetic_data.csv')
df_ids = pd.read_csv('IDS_mapping.csv')

In [4]:
df_diabetic_data.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [5]:
df_ids.head(15)

,admission_type_id,description
0,1,Emergency
1,2,Urgent
2,3,Elective
3,4,Newborn
4,5,Not Available
5,6,NaN
6,7,Trauma Center
7,8,Not Mapped
8,NaN,NaN
9,discharge_disposition_id,description


Per the UCI dataset's definition, an encounter was included if "any kind of diabetes was entered to the system as a diagnosis". This includes drug codes and complications. Checking occurences for Ischemic Heart Disease and Chronic Kidney Disease to check if they are valid inclusions:

In [30]:
ihd = list([str(num) for num in (range(390, 460))])
ckd = list([str(num) for num in (range(580, 630))])

In [39]:
df_diabetic_data["diag_1"][df_diabetic_data["diag_1"].str[:3].isin(ihd)].count() +\
df_diabetic_data["diag_2"][df_diabetic_data["diag_2"].str[:3].isin(ihd)].count() +\
df_diabetic_data["diag_3"][df_diabetic_data["diag_3"].str[:3].isin(ihd)].count()

91619

In [40]:
df_diabetic_data["diag_1"][df_diabetic_data["diag_1"].str[:3].isin(ckd)].count() +\
df_diabetic_data["diag_2"][df_diabetic_data["diag_2"].str[:3].isin(ckd)].count() +\
df_diabetic_data["diag_3"][df_diabetic_data["diag_3"].str[:3].isin(ckd)].count()

19392

These 2 diagnoses can be selected for consideration due to significant presence in the cohort

In [42]:
condition = df_diabetic_data["diag_1"].str[:3].isin(ihd) | \
            df_diabetic_data["diag_2"].str[:3].isin(ihd) | \
            df_diabetic_data["diag_3"].str[:3].isin(ihd) | \
            df_diabetic_data["diag_1"].str[:3].isin(ckd) | \
            df_diabetic_data["diag_2"].str[:3].isin(ckd) | \
            df_diabetic_data["diag_3"].str[:3].isin(ckd)
cohort = df_diabetic_data[condition]

In [45]:
print(len(df_diabetic_data), len(cohort))

101766 68060


In [46]:
cohort.to_csv('cohort.csv', index=False)

In [27]:
cohort = pd.read_csv('cohort.csv')
cohort.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
1,35754,82637451,Caucasian,Male,[50-60),?,2,1,2,3,...,No,Steady,No,No,No,No,No,No,Yes,>30
2,55842,84259809,Caucasian,Male,[60-70),?,3,1,2,4,...,No,Steady,No,No,No,No,No,Ch,Yes,NO
3,63768,114882984,Caucasian,Male,[70-80),?,1,1,7,5,...,No,No,No,No,No,No,No,No,Yes,>30
4,12522,48330783,Caucasian,Female,[80-90),?,2,1,4,13,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


Task 1: Use an ensemble model (stacked generalization / super learner) to decide if patient is at risk of readmission:

- Feature Engineering

In [8]:
cohort["medical_specialty"].value_counts()

medical_specialty
?                         33764
InternalMedicine           9781
Cardiology                 4988
Emergency/Trauma           4901
Family/GeneralPractice     4872
                          ...  
Proctology                    1
Psychiatry-Addictive          1
Dentistry                     1
Neurophysiology               1
Pediatrics-Pulmonology        1
Name: count, Length: 64, dtype: int64

In [28]:
# Checking how many missing values for weight
cohort["weight"][cohort["weight"]=='?'].count()

65875

In [29]:
# Dropping weight column since it has too many missing values
cohort = cohort.drop(columns=["weight"], axis=1)
cohort.head()

,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,500364,82442376,Caucasian,Male,[30-40),1,1,7,2,?,...,No,Up,No,No,No,No,No,Ch,Yes,NO
1,35754,82637451,Caucasian,Male,[50-60),2,1,2,3,?,...,No,Steady,No,No,No,No,No,No,Yes,>30
2,55842,84259809,Caucasian,Male,[60-70),3,1,2,4,?,...,No,Steady,No,No,No,No,No,Ch,Yes,NO
3,63768,114882984,Caucasian,Male,[70-80),1,1,7,5,?,...,No,No,No,No,No,No,No,No,Yes,>30
4,12522,48330783,Caucasian,Female,[80-90),2,1,4,13,?,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [30]:
cohort.columns

Index(['encounter_id', 'patient_nbr', 'race', 'gender', 'age',
       'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
       'time_in_hospital', 'payer_code', 'medical_specialty',
       'num_lab_procedures', 'num_procedures', 'num_medications',
       'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1',
       'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult',
       'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
       'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
       'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
       'tolazamide', 'examide', 'citoglipton', 'insulin',
       'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted'],
      dtype='object')

In [34]:
print(cohort.isnull().sum())

encounter_id                    0
patient_nbr                     0
race                            0
gender                          0
age                             0
admission_type_id               0
discharge_disposition_id        0
admission_source_id             0
time_in_hospital                0
payer_code                      0
medical_specialty               0
num_lab_procedures              0
num_procedures                  0
num_medications                 0
number_outpatient               0
number_emergency                0
number_inpatient                0
diag_1                          0
diag_2                          0
diag_3                          0
number_diagnoses                0
max_glu_serum               64718
A1Cresult                   57144
metformin                       0
repaglinide                     0
nateglinide                     0
chlorpropamide                  0
glimepiride                     0
acetohexamide                   0
glipizide     

In [35]:
# Dropping max_glu_serum and A1Cresult since they have too many missing values
cohort = cohort.drop(columns=["max_glu_serum", "A1Cresult"], axis=1)
cohort.head()

,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,500364,82442376,Caucasian,Male,[30-40),1,1,7,2,?,...,No,Up,No,No,No,No,No,Ch,Yes,NO
1,35754,82637451,Caucasian,Male,[50-60),2,1,2,3,?,...,No,Steady,No,No,No,No,No,No,Yes,>30
2,55842,84259809,Caucasian,Male,[60-70),3,1,2,4,?,...,No,Steady,No,No,No,No,No,Ch,Yes,NO
3,63768,114882984,Caucasian,Male,[70-80),1,1,7,5,?,...,No,No,No,No,No,No,No,No,Yes,>30
4,12522,48330783,Caucasian,Female,[80-90),2,1,4,13,?,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [38]:
for col in cohort.columns:
    missing_values = cohort[col][cohort[col] == '?'].count()
    if missing_values > 0:
        print(f"{col}: {missing_values} missing values")

race: 1412 missing values
payer_code: 27819 missing values
medical_specialty: 33764 missing values
diag_1: 13 missing values
diag_2: 34 missing values
diag_3: 241 missing values


In [39]:
# Deleting payer_code and medical_specialty
cohort = cohort.drop(columns=["payer_code", "medical_specialty"], axis=1)

In [40]:
# replacing '?' with 'unknown' for the remaining columns
cohort = cohort.replace('?', 'unknown')

In [41]:
print(cohort["encounter_id"].nunique(), cohort["patient_nbr"].nunique())

68060 50353


In [42]:
cohort["age"].value_counts()

age
[70-80)     19304
[60-70)     15737
[80-90)     13057
[50-60)     10948
[40-50)      4976
[90-100)     2162
[30-40)      1425
[20-30)       385
[10-20)        63
[0-10)          3
Name: count, dtype: int64

In [43]:
cohort.columns

Index(['encounter_id', 'patient_nbr', 'race', 'gender', 'age',
       'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
       'time_in_hospital', 'num_lab_procedures', 'num_procedures',
       'num_medications', 'number_outpatient', 'number_emergency',
       'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses',
       'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
       'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
       'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
       'tolazamide', 'examide', 'citoglipton', 'insulin',
       'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted'],
      dtype='object')

In [46]:
age_map = {
    '[0-10)': 5, '[10-20)': 15, '[20-30)': 25, '[30-40)': 35, '[40-50)': 45,
    '[50-60)': 55, '[60-70)': 65, '[70-80)': 75, '[80-90)': 85, '[90-100)': 95
}
cohort['age_numeric'] = cohort['age'].map(age_map)


In [45]:
for col in ['number_outpatient', 'number_emergency', 'number_inpatient']:
    cohort[f'{col}_log'] = np.log1p(cohort[col])

In [49]:
cohort["gender_binary"] = cohort["gender"].map({"Female": 0, "Male": 1, 'Unknown/Invalid': -1}).fillna(-1)

In [52]:
cohort = pd.get_dummies(cohort, columns=["race"], prefix="race", drop_first=True)

In [53]:
# Map complex strings into primary ICD-9 physical system blocks
def collapse_icd9(code):
    if pd.isna(code) or code == "?":
        return "Unknown"
    code_str = str(code).split(".")[0].strip()

    # Capture numeric diagnostic groups
    if code_str.isdigit():
        val = int(code_str)
        if 390 <= val <= 459 or val == 785:
            return "Circulatory"
        elif 460 <= val <= 519 or val == 786:
            return "Respiratory"
        elif 520 <= val <= 579 or val == 787:
            return "Digestive"
        elif val == 250:
            return "Diabetes"
        elif 800 <= val <= 999:
            return "Injury"
        elif 710 <= val <= 739:
            return "Musculoskeletal"
        elif 580 <= val <= 629 or val == 788:
            return "Genitourinary"
        elif 140 <= val <= 239:
            return "Neoplasms"
        else:
            return "Other_Numeric"
    else:
        # Capture character alpha codes (V and E codes)
        if code_str.startswith("V") or code_str.startswith("E"):
            return "External_Causes"
        return "Unknown"


# Convert the 3 diagnosis columns to system category strings
for diag_col in ["diag_1", "diag_2", "diag_3"]:
    cohort[f"{diag_col}_group"] = cohort[diag_col].apply(collapse_icd9)

# One-hot encode the collapsed diagnosis matrices
cohort = pd.get_dummies(
    cohort,
    columns=["diag_1_group", "diag_2_group", "diag_3_group"],
    drop_first=True,
)

In [54]:
med_dosage_map = {"No": 0, "Steady": 2, "Down": 1, "Up": 3}

medication_columns = [
    "metformin",
    "repaglinide",
    "nateglinide",
    "chlorpropamide",
    "glimepiride",
    "acetohexamide",
    "glipizide",
    "glyburide",
    "tolbutamide",
    "pioglitazone",
    "rosiglitazone",
    "acarbose",
    "miglitol",
    "troglitazone",
    "tolazamide",
    "examide",
    "citoglipton",
    "insulin",
    "glyburide-metformin",
    "glipizide-metformin",
    "glimepiride-pioglitazone",
    "metformin-rosiglitazone",
    "metformin-pioglitazone",
]

for med in medication_columns:
    cohort[f"{med}_encoded"] = cohort[med].map(med_dosage_map).fillna(0)


# --- 5. Global Clinical Metadata Booleans ---
cohort["change_binary"] = cohort["change"].map({"No": 0, "Ch": 1}).fillna(0)
cohort["diabetesMed_binary"] = (
    cohort["diabetesMed"].map({"No": 0, "Yes": 1}).fillna(0)
)

In [55]:
# Force ascending time serialization
cohort = cohort.sort_values(by=["patient_nbr", "encounter_id"]).reset_index(drop=True)

# Separate the absolute first tracking baseline encounter from the remainder matrix
first_encounters = cohort.drop_duplicates(subset=["patient_nbr"], keep="first").copy()
subsequent_encounters = cohort[~cohort["encounter_id"].isin(first_encounters["encounter_id"])].copy()


In [56]:
print(f"Original Database Matrix Elements: {cohort.shape}")
print(f"Unique Patient Feature Rows (Train Base Pool): {first_encounters.shape[0]}")
print(f"Tracking Array Rows (Subsequent Events Pool): {subsequent_encounters.shape[0]}")

Original Database Matrix Elements: (68060, 109)
Unique Patient Feature Rows (Train Base Pool): 50353
Tracking Array Rows (Subsequent Events Pool): 17707


0    55
1    85
2    35
3    65
4    65
Name: age_numeric, dtype: int64

In [65]:
# Changing the readmission target into a survival format with time and event indicators
def structure_survival_targets(row):
    status = row['readmitted']
    if status == '<30':
        return 15.0, 1.0  # Time=15, Event=1 (Observed early)
    elif status == '>30':
        return 45.0, 1.0  # Time=45, Event=1 (Observed late)
    else:
        return 60.0, 0.0  # Time=60, Event=0 (Right-censored at tracking cap)
    
survival_data = first_encounters.apply(structure_survival_targets, axis=1, result_type='expand')
first_encounters['duration'] = survival_data[0]
first_encounters['event'] = survival_data[1]

# Drop target and index metadata to form your clean feature matrix X
X = first_encounters.drop(columns=['encounter_id', 'patient_nbr', 'readmitted', 'duration', 'event', 'age', 'gender'])

# format target as a structured array required by scikit-survival
y_survival = np.array(
    list(zip(first_encounters['event'].astype(bool), first_encounters['duration'])),
    dtype=[('Status', '?'), ('Survival_in_days', '<f8')]
)

In [66]:
race_cols = [c for c in first_encounters.columns if c.startswith('race_') and c != 'race']
diag_cols = [c for c in first_encounters.columns if '_group_' in c]
med_cols = [c for c in first_encounters.columns if c.endswith('_encoded')]

columns_to_export = [
    # Metadata & Tracking Identifiers
    'encounter_id', 
    'patient_nbr', 
    
    # Original Source Fields (Great for verification)
    'age', 
    'gender', 
    'readmitted',
    
    # Target Metrics for Survival Analysis
    'duration', 
    'event',
    
    # Engineered Demographics & Utilization Metrics
    'age_numeric', 
    'gender_binary',
    'number_outpatient_log', 
    'number_emergency_log', 
    'number_inpatient_log',
    'time_in_hospital', 
    'num_lab_procedures', 
    'num_procedures', 
    'num_medications', 
    'number_diagnoses',
    'change_binary', 
    'diabetesMed_binary'
] + race_cols + diag_cols + med_cols

print(len(columns_to_export), "\n", len(cohort.columns))

77 
 109


In [68]:
first_encounters[columns_to_export].to_csv('first_encounters_processed.csv', index=False)
subsequent_encounters.to_csv('subsequent_encounters_processed.csv', index=False)